In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
import lightgbm as lgb

RANDOM_STATE = 42

app = pd.read_csv("../data/raw/application_train.csv")

X = app.drop(columns=["TARGET", "SK_ID_CURR"])
y = app["TARGET"]

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Development set:", X_dev.shape)
print("Test set:", X_test.shape)
print("Development default rate:", round(y_dev.mean(), 4))
print("Test default rate:", round(y_test.mean(), 4))

Development set: (246008, 120)
Test set: (61503, 120)
Development default rate: 0.0807
Test default rate: 0.0807


In [2]:
def prep_for_lgbm(df):
    df = df.copy()

    df["DAYS_EMPLOYED_ANOM"] = (
        df["DAYS_EMPLOYED"] == 365243
    ).astype("int8")

    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(
        365243, np.nan
    )

    for col in df.select_dtypes(exclude="number").columns:
        df[col] = df[col].astype("category")

    for col in df.select_dtypes(include="float64").columns:
        df[col] = df[col].astype("float32")

    return df

In [3]:
model = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    verbose=-1,
    n_jobs=2
)

In [4]:
print(
    "AMT_INCOME_TOTAL == 0:",
    (X_dev["AMT_INCOME_TOTAL"] == 0).sum()
)

print(
    "CNT_FAM_MEMBERS == 0:",
    (X_dev["CNT_FAM_MEMBERS"] == 0).sum()
)

AMT_INCOME_TOTAL == 0: 0
CNT_FAM_MEMBERS == 0: 0


In [5]:
def add_ratios(df):
    df = df.copy()

    df["CREDIT_INCOME_RATIO"] = (
        df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
    )

    df["ANNUITY_INCOME_RATIO"] = (
        df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    )

    df["CREDIT_TERM"] = (
        df["AMT_ANNUITY"] / df["AMT_CREDIT"]
    )

    df["GOODS_CREDIT_RATIO"] = (
        df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"]
    )

    df["INCOME_PER_PERSON"] = (
        df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]
    )

    return df

In [6]:
Xf_ratios = add_ratios(X_dev)

print(
    "Infinite values:",
    np.isinf(
        Xf_ratios.select_dtypes(include="number")
    ).sum().sum()
)

print(
    "New columns:",
    [
        "CREDIT_INCOME_RATIO",
        "ANNUITY_INCOME_RATIO",
        "CREDIT_TERM",
        "GOODS_CREDIT_RATIO",
        "INCOME_PER_PERSON"
    ]
)

Infinite values: 0
New columns: ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM', 'GOODS_CREDIT_RATIO', 'INCOME_PER_PERSON']


In [7]:
Xf = prep_for_lgbm(add_ratios(X_dev))

aucs = cross_val_score(
    model,
    Xf,
    y_dev,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

print(f"+ Ratios: {aucs.mean():.4f} ± {aucs.std():.4f}")

+ Ratios: 0.7601 ± 0.0009


In [8]:
def add_time(df):
    df = df.copy()

    df["AGE_YEARS"] = df["DAYS_BIRTH"] / -365

    df["YEARS_EMPLOYED"] = df["DAYS_EMPLOYED"] / -365

    df["EMPLOYED_AGE_RATIO"] = (
        df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]
    )

    df["YEARS_REGISTERED"] = (
        df["DAYS_REGISTRATION"] / -365
    )

    df["YEARS_ID_PUBLISHED"] = (
        df["DAYS_ID_PUBLISH"] / -365
    )

    return df

In [9]:
Xf_time = prep_for_lgbm(add_time(X_dev))

print("Shape:", Xf_time.shape)

print(
    "Infinite values:",
    np.isinf(
        Xf_time.select_dtypes(include="number")
    ).sum().sum()
)

print(
    "NaN values:",
    Xf_time.isna().sum().sum()
)

Shape: (246008, 126)
Infinite values: 0
NaN values: 7362588


In [10]:
aucs = cross_val_score(
    model,
    Xf_time,
    y_dev,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

print(f"+ Time: {aucs.mean():.4f} ± {aucs.std():.4f}")

+ Time: 0.7534 ± 0.0012


In [11]:
def add_ext(df):
    df = df.copy()

    ext = df[
        ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    ]

    df["EXT_MEAN"] = ext.mean(axis=1)
    df["EXT_MIN"] = ext.min(axis=1)
    df["EXT_MAX"] = ext.max(axis=1)
    df["EXT_STD"] = ext.std(axis=1)
    df["EXT_COUNT"] = ext.notna().sum(axis=1)

    df["EXT_PROD"] = ext.fillna(ext.mean()).prod(axis=1)

    return df

In [12]:
Xf_ext = prep_for_lgbm(add_ext(X_dev))

print("Shape:", Xf_ext.shape)

print(
    "Infinite values:",
    np.isinf(
        Xf_ext.select_dtypes(include="number")
    ).sum().sum()
)

print(
    "NaN values:",
    Xf_ext.isna().sum().sum()
)

Shape: (246008, 127)
Infinite values: 0
NaN values: 7392420


In [13]:
aucs = cross_val_score(
    model,
    Xf_ext,
    y_dev,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

print(f"+ EXT_SOURCE: {aucs.mean():.4f} ± {aucs.std():.4f}")

+ EXT_SOURCE: 0.7533 ± 0.0007


In [14]:
def add_flags(df):
    df = df.copy()

    doc_cols = [
        c for c in df.columns
        if c.startswith("FLAG_DOCUMENT_")
    ]

    df["DOC_COUNT"] = df[doc_cols].sum(axis=1)

    contact = [
        "FLAG_MOBIL",
        "FLAG_EMP_PHONE",
        "FLAG_WORK_PHONE",
        "FLAG_CONT_MOBILE",
        "FLAG_PHONE",
        "FLAG_EMAIL",
    ]

    contact_cols = [c for c in contact if c in df.columns]

    df["CONTACT_COUNT"] = df[contact_cols].sum(axis=1)

    return df

In [15]:
Xf_flags = prep_for_lgbm(add_flags(X_dev))

print("Shape:", Xf_flags.shape)

print(
    "Infinite values:",
    np.isinf(
        Xf_flags.select_dtypes(include="number")
    ).sum().sum()
)

print(
    "NaN values:",
    Xf_flags.isna().sum().sum()
)

Shape: (246008, 123)
Infinite values: 0
NaN values: 7362588


In [16]:
aucs = cross_val_score(
    model,
    Xf_flags,
    y_dev,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

print(f"+ Flags: {aucs.mean():.4f} ± {aucs.std():.4f}")

+ Flags: 0.7536 ± 0.0019


In [17]:
def add_all_features(df):
    df = df.copy()

    df = add_ratios(df)
    df = add_time(df)
    df = add_ext(df)
    df = add_flags(df)

    return df

In [18]:
Xf_all = add_all_features(X_dev)

print("Shape before LightGBM prep:", Xf_all.shape)

Xf_all = prep_for_lgbm(Xf_all)

print("Shape after LightGBM prep:", Xf_all.shape)

print(
    "Infinite values:",
    np.isinf(
        Xf_all.select_dtypes(include="number")
    ).sum().sum()
)

Shape before LightGBM prep: (246008, 138)
Shape after LightGBM prep: (246008, 139)
Infinite values: 0


In [19]:
aucs = cross_val_score(
    model,
    Xf_all,
    y_dev,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

print(f"+ All features: {aucs.mean():.4f} ± {aucs.std():.4f}")

+ All features: 0.7602 ± 0.0007


In [20]:
X_lean = prep_for_lgbm(add_ratios(X_dev))

aucs = cross_val_score(
    model,
    X_lean,
    y_dev,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

print(f"Lean: {aucs.mean():.4f} ± {aucs.std():.4f}")

Lean: 0.7601 ± 0.0009


In [21]:
m = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_estimators=300,
    learning_rate=0.05,
    verbose=-1,
    n_jobs=2
).fit(X_lean, y_dev)

imp = pd.DataFrame({
    "feature": X_lean.columns,
    "gain": m.booster_.feature_importance("gain")
})

imp = imp.sort_values("gain", ascending=False)

imp.head(20)

,feature,gain
41,EXT_SOURCE_3,65445.033008
40,EXT_SOURCE_2,53501.556474
38,ORGANIZATION_TYPE,20875.123032
39,EXT_SOURCE_1,19698.230735
122,CREDIT_TERM,19575.546041
123,GOODS_CREDIT_RATIO,10141.029792
16,DAYS_EMPLOYED,9437.781308
15,DAYS_BIRTH,6641.499102
7,AMT_ANNUITY,4653.911314
19,OWN_CAR_AGE,4445.657880


In [24]:
fe_results = [
    {"features": "baseline", "cv_auc": 0.7536, "cv_std": 0.0019},
    {"features": "+ ratios", "cv_auc": 0.7601, "cv_std": 0.0009},
    {"features": "+ time", "cv_auc": 0.7534, "cv_std": 0.0012},
    {"features": "+ ext", "cv_auc": 0.7533, "cv_std": 0.0007},
    {"features": "+ flags", "cv_auc": 0.7536, "cv_std": 0.0019},
    {"features": "+ all features", "cv_auc": 0.7602, "cv_std": 0.0007},
]

pd.DataFrame(fe_results).to_csv(
    "../reports/feature_experiments.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [25]:
pd.read_csv("../reports/feature_experiments.csv")

,features,cv_auc,cv_std
0,baseline,0.7536,0.0019
1,+ ratios,0.7601,0.0009
2,+ time,0.7534,0.0012
3,+ ext,0.7533,0.0007
4,+ flags,0.7536,0.0019
5,+ all features,0.7602,0.0007


## Feature Engineering Findings

1. Ratio features produced the only clear improvement among the engineered
   feature groups tested. Adding five ratios increased CV ROC-AUC from
   0.7536 ± 0.0019 to 0.7601 ± 0.0009.

2. The ratio features were:
   - CREDIT_INCOME_RATIO
   - ANNUITY_INCOME_RATIO
   - CREDIT_TERM
   - GOODS_CREDIT_RATIO
   - INCOME_PER_PERSON

3. Time-derived features did not improve performance. The model achieved
   0.7534 ± 0.0012 compared with the 0.7536 baseline.

4. EXT_SOURCE combination features also did not improve performance.
   Although EXT_SOURCE_1, EXT_SOURCE_2 and EXT_SOURCE_3 were among the
   strongest individual predictors, their engineered combinations achieved
   0.7533 ± 0.0007. The raw EXT_SOURCE features were already available to
   the model, so these combinations added little new information.

5. Document and contact count features did not change performance:
   0.7536 ± 0.0019.

6. Combining all engineered feature groups produced 0.7602 ± 0.0007.
   This was only 0.0001 higher than ratios alone. The additional 17
   engineered features therefore provided negligible incremental value.

7. The lean feature set was verified at 0.7601 ± 0.0009 and was selected
   instead of the larger combined set because it achieves essentially the
   same performance with fewer features.

8. Feature importance from the lean model showed that several engineered
   ratios were among the most important features:
   CREDIT_TERM ranked 5th, GOODS_CREDIT_RATIO ranked 6th,
   ANNUITY_INCOME_RATIO ranked 18th, and CREDIT_INCOME_RATIO ranked 20th
   by LightGBM gain.

9. The results suggest that feature engineering is most useful for tree
   models when it creates relationships that are not easily represented
   by the original features. Ratio features introduce such relationships,
   while transformations such as converting DAYS_BIRTH into AGE_YEARS
   mainly restate information already available to the tree.

10. The application-only lean LightGBM model, with the five ratio features
    and DAYS_EMPLOYED anomaly handling, is the current benchmark for the
    next stage of the project: adding information from the supporting
    credit-history tables.